###Delta History

In [0]:
%sql
DESCRIBE HISTORY finance_dev.silver_customers_scd2;

###Time Travel

Delta Lake allows querying historical snapshots of a table using VERSION AS OF or TIMESTAMP AS OF. This is useful for audit, debugging, recovery, and reproducible analytics.

In [0]:
%sql
select * from finance_dev.silver_customers_scd2 version as of 3

###Partition pruning

The query scans only the partition for 2025-07-01 instead of the entire Bronze table, reducing file reads and improving performance.

In [0]:
%sql
DESCRIBE DETAIL finance_dev.bronze_transactions

In [0]:
%sql
EXPLAIN
SELECT *
FROM finance_dev.bronze_transactions
WHERE TransactionDate = '2025-07-01';

###OPTIMIZE + ZORDER

OPTIMIZE compacts many small Delta files into larger files. ZORDER colocates related values (CustomerID, TransactionDate) to reduce file scanning for selective queries.

In [0]:
%sql
OPTIMIZE finance_dev.silver_transactions_enriched
ZORDER BY (CustomerID, TransactionDate);

###Broadcast join

Small dimension tables are broadcast to executors, avoiding expensive shuffle joins and improving performance in star-schema enrichments.

In [0]:
from pyspark.sql.functions import broadcast

txn = spark.table("finance_dev.silver_transactions_staging")
acc = spark.table("finance_dev.silver_accounts")
fx = spark.table("finance_dev.silver_exchange_rates")

broadcast_demo = (
    txn.join(broadcast(acc), "AccountID", "left")
       .join(broadcast(fx), "Currency", "left")
)

display(broadcast_demo.limit(5))

###VACUUM

VACUUM removes obsolete data files that are no longer referenced by the Delta transaction log. Retaining 168 hours (7 days) preserves recent time-travel capability.

In [0]:
%sql
--VACUUM finance_dev.silver_customers_scd2 RETAIN 168 HOURS;

###Change Data Feed (no implementation)


ALTER TABLE finance_dev.silver_customers_scd2
SET TBLPROPERTIES (delta.enableChangeDataFeed = true)

Explanation:

exposes inserts/updates/deletes between versions

useful for incremental downstream processing

not implemented in this project